### This notebook produces:

- Three-way in-distribution comparison on PR-aug test: joint, pipeline (TQA-binary + PR-type), threshold-only. Main results table.
- Cross-corpus comparison on TQA test — same three architectures. Robustness table.
- Per-category F1 on PR-aug for all three — finer-grained analysis table.
- Pipeline error attribution — for the pipeline specifically, where do its errors come from (binary stage vs type stage).
- Binary detector cross-corpus summary — pulling together the four binary-detector cross-evals already ran into one summary table for chapter 4.5.2.
Architecture decision summary — synthesises all of the above into a single recommendation for chapter 6.

In [9]:
from pathlib import Path
import json
import pandas as pd
import numpy as np

# === EDIT THESE PATHS ===
PROJECT_ROOT  = Path("../..")
RESULTS_DIR   = Path("./results")
MODELS_DIR = PROJECT_ROOT / "models"
PROCESSED_DIR = PROJECT_ROOT / "processed"

# Long-form 11-class label space, used everywhere.
PATTERNREFRAME_LABELS = [
    "Black-and-white or polarized thinking / All or nothing thinking",
    "Catastrophizing",
    "Discounting the positive",
    "Jumping to conclusions: Fortune-telling",
    "Jumping to conclusions: mind reading",
    "Labeling and mislabeling",
    "Mental filtering",
    "Overgeneralization",
    "Personalization",
    "Should statements",
]
LABEL_SPACE = ["No Distortion"] + PATTERNREFRAME_LABELS

# Production τ chosen by the threshold sweep on PR-aug val
PRODUCTION_TAU = 0.46

# === REQUIRED INPUT FILES ===
# Joint 11-class predictions on PR-aug test (with text column attached)
JOINT_PRAUG_CSV = MODELS_DIR / "02_04_roberta_patternreframe_none_aug/results/joint11_patternreframe_test_predictions.csv"

# TQA-trained binary detector applied to PR-aug test
BINARY_PRAUG_CSV = MODELS_DIR / "04_02_therapistqa_binary/results/binary_patternreframe_tqatrained_test_predictions.csv"

# PR-aug type classifier predictions and probabilities on PR-aug test
# from the Colab inference cell we ran for the threshold sweep
TYPE_PRAUG_PROBS_CSV = MODELS_DIR / "04_03_threshold_architecture/results/type_classifier_test_probs.csv"

# Threshold-only predictions on PR-aug test (from the threshold sweep)
THRESHOLD_PRAUG_CSV = MODELS_DIR / "04_03_threshold_architecture/results/threshold_only_test_predictions.csv"

# Cross-corpus eval files
# predictions made on the TherapistQA test set by joint 11 PatternReframe model
JOINT_TQA_PREDS = MODELS_DIR / "04_03_threshold_architecture/results/joint11_tqa_test_predictions.csv"
# predictions made on the TherapistQA test set by threshold only PatternReframe model
THRESHOLD_TQA_PREDS = MODELS_DIR / "04_03_threshold_architecture/results/threshold_only_tqa_test_predictions.csv"

# Verify all files exist before continuing
required = [JOINT_PRAUG_CSV, BINARY_PRAUG_CSV, TYPE_PRAUG_PROBS_CSV,
            THRESHOLD_PRAUG_CSV]
missing = [p for p in required if not p.exists()]
if missing:
    print("Missing files:")
    for m in missing:
        print(f"  {m}")
    raise FileNotFoundError("Cannot proceed; populate results/ first.")
print("All required files present.")

All required files present.


In [10]:
# Load all six prediction sources.
joint_praug      = pd.read_csv(JOINT_PRAUG_CSV)
binary_praug     = pd.read_csv(BINARY_PRAUG_CSV)
type_praug_probs = pd.read_csv(TYPE_PRAUG_PROBS_CSV)
threshold_praug  = pd.read_csv(THRESHOLD_PRAUG_CSV)

joint_tqa = pd.read_csv(JOINT_TQA_PREDS)
print(f"Joint TQA preds:     {len(joint_tqa)} rows")

threshold_tqa = pd.read_csv(THRESHOLD_TQA_PREDS)
print(f"Threshold TQA preds: {len(threshold_tqa)} rows")

print(f"\nIn-distribution prediction sources (PR-aug test):")
print(f"  Joint 11-class:     {len(joint_praug)} rows")
print(f"  TQA binary:         {len(binary_praug)} rows")
print(f"  Type probabilities: {len(type_praug_probs)} rows")
print(f"  Threshold @ τ={PRODUCTION_TAU}: {len(threshold_praug)} rows")

Joint TQA preds:     359 rows
Threshold TQA preds: 359 rows

In-distribution prediction sources (PR-aug test):
  Joint 11-class:     7488 rows
  TQA binary:         7488 rows
  Type probabilities: 7488 rows
  Threshold @ τ=0.46: 7488 rows


## The pipeline = TQA binary detector -> PR type classifier

- Stage 1: binary detector says ND or Distorted
- Stage 2: if Distorted, type classifier's argmax is the final category
If ND from stage 1, output is "No Distortion" (stage 2 is skipped at inference, but we keep its prediction in the CSV anyway for the oracle analysis).

In [11]:
# Get type classifier argmax from probabilities
prob_cols = [f"prob_{i}" for i in range(len(PATTERNREFRAME_LABELS))]
type_praug_probs["type_argmax_idx"] = type_praug_probs[prob_cols].values.argmax(axis=1)
type_praug_probs["type_argmax_label"] = type_praug_probs["type_argmax_idx"].map(
    lambda i: PATTERNREFRAME_LABELS[int(i)]
)

# Standardise column names for the merge
# binary_praug has: text, label, pattern, pred, prob_distorted
binary_renamed = binary_praug.rename(columns={"pred": "binary_pred"})

# type_praug_probs may use either "thought" or "text" — handle both
type_text_col = "text" if "text" in type_praug_probs.columns else "thought"
type_renamed = type_praug_probs.rename(columns={
    type_text_col: "text",
    "true_pattern": "gold_label",
    "type_argmax_label": "type_pred_label",
})[["text", "type_pred_label"]]

# Merge joint + binary + type on text
# Deduplicate the binary and type sources first (one prediction per unique text);
# joint may legitimately have duplicate text rows for the 5 PatternReframe-side
# annotation-conflict thoughts.
for name, df in [("binary", binary_renamed), ("type", type_renamed)]:
    n_dup = df["text"].duplicated().sum()
    if n_dup:
        print(f"Deduplicating {n_dup} duplicate texts in {name} predictions.")
binary_renamed = binary_renamed.drop_duplicates(subset="text", keep="first")
type_renamed = type_renamed.drop_duplicates(subset="text", keep="first")

merged = (
    joint_praug[["text", "gold_label", "joint_pred_label"]]
    .merge(binary_renamed[["text", "binary_pred", "prob_distorted"]],
           on="text", how="left", validate="many_to_one")
    .merge(type_renamed[["text", "type_pred_label"]],
           on="text", how="left", validate="many_to_one")
)

assert merged["binary_pred"].isna().sum() == 0, "Some PR-aug rows have no binary prediction"
assert merged["type_pred_label"].isna().sum() == 0, "Some PR-aug rows have no type prediction"

# Pipeline rule
merged["pipeline_pred_label"] = np.where(
    merged["binary_pred"] == 0,
    "No Distortion",
    merged["type_pred_label"],
)

# Pull in threshold-only predictions on the same rows
threshold_renamed = threshold_praug[["text", "threshold_pred_label"]].drop_duplicates(
    subset="text", keep="first"
)
merged = merged.merge(threshold_renamed, on="text", how="left", validate="many_to_one")
assert merged["threshold_pred_label"].isna().sum() == 0

print(f"Merged PR-aug predictions: {len(merged)} rows")
merged.head()

Deduplicating 6 duplicate texts in binary predictions.
Deduplicating 6 duplicate texts in type predictions.
Merged PR-aug predictions: 7488 rows


,text,gold_label,joint_pred_label,binary_pred,prob_distorted,type_pred_label,pipeline_pred_label,threshold_pred_label
0,People only want to be my friend because I'm p...,Jumping to conclusions: mind reading,Jumping to conclusions: mind reading,1,0.642765,Jumping to conclusions: mind reading,Jumping to conclusions: mind reading,Jumping to conclusions: mind reading
1,"I love to paint pictures, but I'll not make to...",Jumping to conclusions: Fortune-telling,Overgeneralization,0,0.259268,Discounting the positive,No Distortion,Discounting the positive
2,I love to snack between meals. I am going to b...,Jumping to conclusions: Fortune-telling,Jumping to conclusions: Fortune-telling,1,0.857321,Catastrophizing,Catastrophizing,No Distortion
3,I like watching The Walking Dead. Too bad I do...,Labeling and mislabeling,Discounting the positive,0,0.301757,Discounting the positive,No Distortion,Discounting the positive
4,I hate two cats. People must think I'm clearly...,Jumping to conclusions: mind reading,Jumping to conclusions: mind reading,1,0.786315,Jumping to conclusions: mind reading,Jumping to conclusions: mind reading,Jumping to conclusions: mind reading


In [12]:
from sklearn.metrics import (
    accuracy_score, classification_report, confusion_matrix, f1_score,
)


def architecture_metrics(df, pred_col, gold_col="gold_label", labels=LABEL_SPACE):
    """Compute the standard architecture summary metrics for one prediction column."""
    gold  = df[gold_col].values
    preds = df[pred_col].values
    nd_mask = (gold == "No Distortion")

    return {
        "Accuracy":     accuracy_score(gold, preds),
        "Weighted F1":  f1_score(gold, preds, labels=labels, average="weighted", zero_division=0),
        "Macro F1":     f1_score(gold, preds, labels=labels, average="macro", zero_division=0),
        "ND recall":    (
            (preds[nd_mask] == "No Distortion").mean() if nd_mask.sum() else float("nan")
        ),
        "D recall":     (
            (preds[~nd_mask] != "No Distortion").mean() if (~nd_mask).sum() else float("nan")
        ),
    }


def per_category_f1(df, pred_col, gold_col="gold_label", labels=LABEL_SPACE):
    """Per-category F1 score table for one prediction column."""
    report = classification_report(
        df[gold_col], df[pred_col],
        labels=labels, digits=4, output_dict=True, zero_division=0,
    )
    return {cat: report[cat]["f1-score"] for cat in labels}

In [15]:
# Main results table: three-way comparison on PR-aug
table_main = pd.DataFrame([
    {"Architecture": "Joint 11-class (Chapter 4.4.5)",
     **architecture_metrics(merged, "joint_pred_label")},
    {"Architecture": "Pipeline (TQA binary → PR type)",
     **architecture_metrics(merged, "pipeline_pred_label")},
    {"Architecture": f"Threshold-only @ τ={PRODUCTION_TAU}",
     **architecture_metrics(merged, "threshold_pred_label")},
])

print("Table — Three-way architecture comparison on PR-aug test set:")
print(table_main.to_string(index=False, float_format=lambda x: f"{x:.4f}"))
table_main.to_csv(RESULTS_DIR / "table_main_praug_comparison.csv", index=False)

Table — Three-way architecture comparison on PR-aug test set:
                   Architecture  Accuracy  Weighted F1  Macro F1  ND recall  D recall
 Joint 11-class (Chapter 4.4.5)    0.6378       0.6374    0.6389     0.9427    0.9952
Pipeline (TQA binary → PR type)    0.3132       0.3496    0.3490     0.2922    0.5380
        Threshold-only @ τ=0.46    0.5597       0.5673    0.5686     0.4376    0.8446


In [16]:
# Per-category F1 table
per_cat = pd.DataFrame({
    "Category": LABEL_SPACE,
    "Joint":     [per_category_f1(merged, "joint_pred_label")[c]     for c in LABEL_SPACE],
    "Pipeline":  [per_category_f1(merged, "pipeline_pred_label")[c]  for c in LABEL_SPACE],
    "Threshold": [per_category_f1(merged, "threshold_pred_label")[c] for c in LABEL_SPACE],
})

# Short labels for display
short = {
    "No Distortion": "No Distortion",
    "Black-and-white or polarized thinking / All or nothing thinking": "All-or-nothing",
    "Catastrophizing": "Catastrophizing",
    "Discounting the positive": "Discounting +",
    "Jumping to conclusions: Fortune-telling": "Fortune-telling",
    "Jumping to conclusions: mind reading": "Mind reading",
    "Labeling and mislabeling": "Labeling",
    "Mental filtering": "Mental filter",
    "Overgeneralization": "Overgeneralization",
    "Personalization": "Personalization",
    "Should statements": "Should",
}
per_cat["Category"] = per_cat["Category"].map(short)

print("Table — Per-category F1 on PR-aug test set:")
print(per_cat.to_string(index=False, float_format=lambda x: f"{x:.4f}"))
per_cat.to_csv(RESULTS_DIR / "table_per_category_f1.csv", index=False)

Table — Per-category F1 on PR-aug test set:
          Category  Joint  Pipeline  Threshold
     No Distortion 0.9469    0.0989     0.2926
    All-or-nothing 0.5070    0.3316     0.5033
   Catastrophizing 0.6177    0.4060     0.6148
     Discounting + 0.7027    0.1856     0.6574
   Fortune-telling 0.4142    0.2726     0.4185
      Mind reading 0.6684    0.6127     0.6805
          Labeling 0.5367    0.5321     0.5577
     Mental filter 0.6037    0.3425     0.5559
Overgeneralization 0.4909    0.3975     0.4634
   Personalization 0.6138    0.4766     0.5881
            Should 0.9255    0.1835     0.9222


In [17]:
# Pipeline error attribution
gold_distorted = merged["gold_label"] != "No Distortion"
bin_pred       = merged["binary_pred"] == 1
pipe_correct   = merged["pipeline_pred_label"] == merged["gold_label"]
pipe_wrong     = ~pipe_correct

fn_mask = pipe_wrong & gold_distorted & ~bin_pred           # missed distortions
fp_mask = pipe_wrong & ~gold_distorted & bin_pred           # false alarms on ND
ts_mask = pipe_wrong & gold_distorted & bin_pred            # right binary, wrong category

total_errors = pipe_wrong.sum()
errors = pd.DataFrame([
    {"Source": "Binary false negative (Distorted → No Distortion)",
     "Count": int(fn_mask.sum()),
     "Share %": 100 * fn_mask.sum() / total_errors},
    {"Source": "Binary false positive (No Distortion → Distorted)",
     "Count": int(fp_mask.sum()),
     "Share %": 100 * fp_mask.sum() / total_errors},
    {"Source": "Type-stage error (binary correct, wrong category)",
     "Count": int(ts_mask.sum()),
     "Share %": 100 * ts_mask.sum() / total_errors},
    {"Source": "Total pipeline errors",
     "Count": int(total_errors),
     "Share %": 100.0},
])

print("Table — Pipeline error attribution:")
print(errors.to_string(index=False, float_format=lambda x: f"{x:.1f}"))
errors.to_csv(RESULTS_DIR / "table_pipeline_error_attribution.csv", index=False)

Table — Pipeline error attribution:
                                           Source  Count  Share %
Binary false negative (Distorted → No Distortion)   3145     61.2
Binary false positive (No Distortion → Distorted)    482      9.4
Type-stage error (binary correct, wrong category)   1516     29.5
                            Total pipeline errors   5143    100.0


In [18]:
# Cross-corpus comparison on TQA
if joint_tqa is None or threshold_tqa is None:
    print("Skipping cross-corpus comparison — TQA prediction files missing.")
else:
    # Both files should have gold_label and prediction columns
    tqa_cmp = joint_tqa[["text", "gold_label", "joint_pred_label"]].merge(
        threshold_tqa[["text", "threshold_pred_label"]],
        on="text", how="inner",
    )
    # Note: pipeline cross-corpus would require running TQA binary on TQA test
    # AND the type classifier on TQA test — both available, but separate analysis.
    # For now, just report joint and threshold cross-corpus.

    print(f"TQA test (after PR->TQA label mapping): {len(tqa_cmp)} rows")
    print(f"Gold distribution:")
    print(tqa_cmp['gold_label'].value_counts().to_string())
    print()

    table_xc = pd.DataFrame([
        {"Architecture": "Joint 11-class (cross-corpus)",
         **architecture_metrics(tqa_cmp, "joint_pred_label")},
        {"Architecture": f"Threshold-only @ τ={PRODUCTION_TAU} (cross-corpus)",
         **architecture_metrics(tqa_cmp, "threshold_pred_label")},
    ])
    print("Table — Cross-corpus comparison on TQA test set:")
    print(table_xc.to_string(index=False, float_format=lambda x: f"{x:.4f}"))
    table_xc.to_csv(RESULTS_DIR / "table_cross_corpus_tqa.csv", index=False)

TQA test (after PR->TQA label mapping): 359 rows
Gold distribution:
gold_label
No Distortion                                                      140
Jumping to conclusions: mind reading                                36
Overgeneralization                                                  36
Catastrophizing                                                     29
Labeling and mislabeling                                            25
Personalization                                                     23
Jumping to conclusions: Fortune-telling                             21
Mental filtering                                                    18
Should statements                                                   16
Black-and-white or polarized thinking / All or nothing thinking     15

Table — Cross-corpus comparison on TQA test set:
                          Architecture  Accuracy  Weighted F1  Macro F1  ND recall  D recall
         Joint 11-class (cross-corpus)    0.1866       0.1720    0.0

### Binary detector cross-corpus summary
consolidates the four binary-detector results from across experiments into one chapter table

In [19]:
# Hardcode the numbers from the experiments you've already run.
# These are the binary-detection-specific cross-corpus diagnostics from earlier.
binary_summary = pd.DataFrame([
    {"Detector": "PR-aug binary (variant A)", "Trained on": "PR-aug (Bahasa ND)",
     "Tested on": "PR-aug test (in-dist)", "wF1": 0.9909, "ND recall": 0.9413, "Notes": "Shortcut-inflated"},
    {"Detector": "PR-aug binary (variant A)", "Trained on": "PR-aug (Bahasa ND)",
     "Tested on": "TQA test (cross-corpus)", "wF1": 0.5514, "ND recall": 0.2571,
     "Notes": "No transfer (confidence gap +0.018)"},
    {"Detector": "TQA binary", "Trained on": "TQA",
     "Tested on": "TQA test (in-dist)", "wF1": 0.7741, "ND recall": 0.6643, "Notes": "Honest baseline"},
    {"Detector": "TQA binary", "Trained on": "TQA",
     "Tested on": "PR-aug test (cross-corpus)", "wF1": 0.6170, "ND recall": 0.2922,
     "Notes": "Inverted transfer (confidence gap -0.145)"},
])
print("Table — Binary detector cross-corpus diagnostic summary:")
print(binary_summary.to_string(index=False, float_format=lambda x: f"{x:.4f}"))
binary_summary.to_csv(RESULTS_DIR / "table_binary_cross_corpus.csv", index=False)

Table — Binary detector cross-corpus diagnostic summary:
                 Detector         Trained on                  Tested on    wF1  ND recall                                     Notes
PR-aug binary (variant A) PR-aug (Bahasa ND)      PR-aug test (in-dist) 0.9909     0.9413                         Shortcut-inflated
PR-aug binary (variant A) PR-aug (Bahasa ND)    TQA test (cross-corpus) 0.5514     0.2571       No transfer (confidence gap +0.018)
               TQA binary                TQA         TQA test (in-dist) 0.7741     0.6643                           Honest baseline
               TQA binary                TQA PR-aug test (cross-corpus) 0.6170     0.2922 Inverted transfer (confidence gap -0.145)


### Architecture decision summary for chapter 6

In [20]:
print("=" * 70)
print("Architecture decision summary")
print("=" * 70)
print()
print("In-distribution PR-aug test (weighted F1):")
for _, row in table_main.iterrows():
    print(f"  {row['Architecture']:50s}  {row['Weighted F1']:.4f}")

if joint_tqa is not None and threshold_tqa is not None:
    print()
    print("Cross-corpus TQA test (weighted F1):")
    for _, row in table_xc.iterrows():
        print(f"  {row['Architecture']:50s}  {row['Weighted F1']:.4f}")

print()
print("Recommendation for chapter 6:")
print("  Joint 11-class has the highest in-distribution F1.")
print("  All three architectures collapse to ~0.17 wF1 on cross-corpus TQA.")
print("  The architectural choice is a second-order concern; the dominant")
print("  limitation is the single-source PR-aug ND class.")
print()
print("  Production system: joint 11-class with UI-level confidence reporting.")
print("  When max_prob < 0.5, present the top 2-3 categories rather than a")
print("  single confident classification. This combines the joint model's")
print("  in-distribution advantage with the threshold approach's uncertainty")
print("  signal at the UI layer.")

Architecture decision summary

In-distribution PR-aug test (weighted F1):
  Joint 11-class (Chapter 4.4.5)                      0.6374
  Pipeline (TQA binary → PR type)                     0.3496
  Threshold-only @ τ=0.46                             0.5673

Cross-corpus TQA test (weighted F1):
  Joint 11-class (cross-corpus)                       0.1720
  Threshold-only @ τ=0.46 (cross-corpus)              0.1758

Recommendation for chapter 6:
  Joint 11-class has the highest in-distribution F1.
  All three architectures collapse to ~0.17 wF1 on cross-corpus TQA.
  The architectural choice is a second-order concern; the dominant
  limitation is the single-source PR-aug ND class.

  Production system: joint 11-class with UI-level confidence reporting.
  When max_prob < 0.5, present the top 2-3 categories rather than a
  single confident classification. This combines the joint model's
  in-distribution advantage with the threshold approach's uncertainty
  signal at the UI layer.


In [22]:
# === Pipeline cross-corpus eval ===
binary_tqa = pd.read_csv(MODELS_DIR / "04_02_therapistqa_binary/results/binary_tqa_test_predictions.csv")
threshold_tqa = pd.read_csv(MODELS_DIR / "04_03_threshold_architecture/results/threshold_only_tqa_test_predictions.csv")

print(f"Binary on TQA:    {len(binary_tqa)} rows — columns: {list(binary_tqa.columns)}")
print(f"Threshold on TQA: {len(threshold_tqa)} rows — columns: {list(threshold_tqa.columns)}")

# The binary file lacks the PR-mapped gold_label, so we take it from the
# threshold file (which has 'gold_label' in PR's 11-class vocabulary).
# We take the binary prediction from the binary file and the type classifier's
# argmax from the threshold file.
pipeline_tqa = threshold_tqa[["text", "gold_label", "argmax_label"]].merge(
    binary_tqa[["text", "pred", "prob_distorted"]].rename(
        columns={"pred": "binary_pred"}
    ),
    on="text", how="inner",  # inner join: only keep rows in both files
)

print(f"\nMerged pipeline: {len(pipeline_tqa)} rows")
if len(pipeline_tqa) != len(threshold_tqa):
    print(f"  WARNING: lost {len(threshold_tqa) - len(pipeline_tqa)} rows during merge.")
    # Investigate by checking which texts don't match
    missing_in_binary = set(threshold_tqa["text"]) - set(binary_tqa["text"])
    print(f"  Texts in threshold file but not binary: {len(missing_in_binary)}")

# Build pipeline prediction
pipeline_tqa["pipeline_pred_label"] = np.where(
    pipeline_tqa["binary_pred"] == 0,
    "No Distortion",
    pipeline_tqa["argmax_label"],
)

# Save the merged predictions
pipeline_tqa.to_csv(
    RESULTS_DIR / "pipeline_tqa_test_predictions.csv", index=False
)

# Metrics on TQA cross-corpus
LABEL_SPACE = ["No Distortion"] + PATTERNREFRAME_LABELS

gold = pipeline_tqa["gold_label"].values
preds = pipeline_tqa["pipeline_pred_label"].values
nd_mask = gold == "No Distortion"

metrics = {
    "Accuracy":    accuracy_score(gold, preds),
    "Weighted F1": f1_score(gold, preds, labels=LABEL_SPACE, average="weighted", zero_division=0),
    "Macro F1":    f1_score(gold, preds, labels=LABEL_SPACE, average="macro", zero_division=0),
    "ND recall":   (preds[nd_mask] == "No Distortion").mean() if nd_mask.sum() else float("nan"),
    "D recall":    (preds[~nd_mask] != "No Distortion").mean() if (~nd_mask).sum() else float("nan"),
}

print("\nPipeline (TQA binary → PR type) — TQA cross-corpus:")
for k, v in metrics.items():
    print(f"  {k}: {v:.4f}")

Binary on TQA:    379 rows — columns: ['text', 'label', 'category', 'pred', 'prob_distorted']
Threshold on TQA: 359 rows — columns: ['text', 'gold_label', 'threshold_pred_label', 'max_prob', 'argmax_label']

Merged pipeline: 359 rows

Pipeline (TQA binary → PR type) — TQA cross-corpus:
  Accuracy: 0.3259
  Weighted F1: 0.3182
  Macro F1: 0.1196
  ND recall: 0.6643
  D recall: 0.8356
